本地大模型、重并发、高IO系统
-  语音RAG系统

**FastAPI 加载大模型的陷阱**

- FastAPI 是基于单线程事件循环（Event Loop）的。
- 如果你直接写一个 async def load_model()，然后在里面跑 AutoModel.from_pretrained(...) 或者 .to("cuda:0")，这些底层 C++ / PyTorch 的代码是纯同步且阻塞的。
- 结果就是： 虽然你用了 FastAPI，但在模型加载的那几十秒里，整个事件循环被“卡死”了，你的 /health 接口依然会超时，前端还是得干等着！

**解决方案：后台任务+线程池（asyncio.to_thread）**
- 要实现真正的“Web 服务秒起，后台悄悄分显存”，我们得利用 FastAPI 最新的 lifespan 机制，配合异步线程池。

In [ ]:
import asyncio
from fastapi import FastAPI
from pydantic import BaseModel
from contextlib import asynccontextmanager
from loguru import logger
# 假设这是你的模型库
# from funasr import AutoModel 

# 1. 定义一个全局状态模型 (用 Pydantic 更规范)
class SystemStatus(BaseModel):
    is_ready: bool = False
    message: str = "服务已启动，准备初始化..."

# 实例化全局状态
status = SystemStatus()
model = None

# 2. 真正的加载逻辑（必须扔进独立线程里）
def load_heavy_model_sync():
    global model
    # 这里的代码是同步阻塞的，比如疯狂读写硬盘、往 GPU 塞数据
    # model = AutoModel(model="...", task="asr", device="cuda:0")
    
    # 模拟漫长的加载过程 (测试时用)
    import time
    time.sleep(15) 
    return "DummyModel" # 返回加载好的模型实例

# 3. 异步后台包装器
async def init_model_background():
    global model, status
    try:
        status.message = "正在分配显存并加载大模型权重，请耐心等待..."
        logger.info(status.message)
        
        # 【全场最核心的一行】
        # 用 to_thread 把阻塞的加载过程扔到后台线程池去跑，绝对不卡死 FastAPI 主服务！
        model = await asyncio.to_thread(load_heavy_model_sync)
        
        status.is_ready = True
        status.message = "模型加载完毕，Nene 语音系统完全就绪！"
        logger.success(status.message)
    except Exception as e:
        status.message = f"❌ 模型加载失败: {str(e)}"
        logger.error(status.message)

# 4. FastAPI 生命周期管理
@asynccontextmanager
async def lifespan(app: FastAPI):
    # 刚启动时：不加 await，直接 create_task 把它像飞镖一样扔出去在后台跑
    # 这样 lifespan 会立刻向下执行，FastAPI 服务瞬间启动！
    task = asyncio.create_task(init_model_background())
    
    yield # 这里代表 Web 服务正在运行的时间段
    
    # 停止服务时：清理显存
    logger.info("服务关闭，正在释放显存...")
    # if model: del model

# 5. 挂载应用
app = FastAPI(lifespan=lifespan)

# 6. 给前端无限轮询的探针接口
@app.get("/health", response_model=SystemStatus)
async def check_health():
    """前端每隔 2 秒请求一次这个接口"""
    return status

# 运行命令：uvicorn main:app --port 5000